<a href="https://colab.research.google.com/github/Levan-Danelia/FRTB/blob/main/FRTB_DRC_Non_Sec.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>



# FRTB ASA: DRC Calculation for Non-Securitisations

This notebook implements the Default Risk Charge (DRC) calculation for non-securitisation positions according to the FRTB Advanced Standardised Approach, following Articles 325w, 325x, and 325y.

We will perform the calculation for three example portfolios:
1. Equity Portfolio
2. Debt Portfolio (assuming Senior)
3. Debt Portfolio (assuming Non-Senior)

The calculation follows these 6 steps:
1. Identify Portfolio Positions
2. Calculate Gross Jump-to-Default (JTD)
3. Calculate Net JTD (Offsetting and Scaling)
4. Apply Default Risk Weights
5. Calculate Hedge Benefit Ratio (WtS)
6. Aggregate Within the Bucket



In [8]:
# Cell 1: Setup and Portfolio Definitions
# Import necessary libraries
import pandas as pd
import numpy as np

# Ensure pandas displays floats nicely without scientific notation and with commas
pd.options.display.float_format = '{:,.0f}'.format

# --- Equity Portfolio Data ---
equity_data = {
    'Bucket': ['Corp', 'Corp', 'Corp', 'Corp'],
    'Issuer': ['Issuer A', 'Issuer A', 'Issuer B', 'Issuer B'],
    'Maturity': [0.25, 0.25, 0.25, 0.25],
    'Asset_Class': ['Equity', 'Equity', 'Equity', 'Equity'],
    'Seniority': [np.nan, np.nan, np.nan, np.nan], # Not applicable for equity
    'Rating': ['Unrated', 'Unrated', 'BBB', 'BBB'],
    'Exposure_VA': [697, -1240, 9843, -2659]
}
equity_df = pd.DataFrame(equity_data)

# --- Debt Portfolio Data ---
debt_data = {
    'Bucket': ['Sovereign', 'Sovereign', 'Sovereign', 'Sovereign', 'Sovereign', 'Sovereign'],
    'Issuer': ['Issuer A', 'Issuer A', 'Issuer B', 'Issuer B', 'Issuer C', 'Issuer C'],
    'Scaled_Maturity_Factor': [0.25, 0.25, 0.77, 1.00, 1.00, 1.00], # Pre-calculated scaling factor based on Article 325x
    'Asset_Class': ['Debt', 'Debt', 'Debt', 'Debt', 'Debt', 'Debt'],
    # Seniority will be assigned per example
    'Rating': ['CCC', 'CCC', 'AA', 'AA', 'AA', 'AA'],
    'Exposure_VA': [-93197, 186394, -1060061, 913941, 499947, -270395]
}
debt_df_base = pd.DataFrame(debt_data)

# Create specific copies for Senior and Non-Senior examples
debt_senior_df = debt_df_base.copy()
debt_senior_df['Seniority'] = 'Senior'

debt_non_senior_df = debt_df_base.copy()
debt_non_senior_df['Seniority'] = 'Non-Senior'

# --- Regulatory Parameters ---
# LGD values (Article 325w)
lgd_map = {
    'Equity': 1.00,        # Article 325w(4)
    'Senior': 0.75,        # Article 325w(3)(b)
    'Non-Senior': 1.00,    # Article 325w(3)(a)
    'Covered Bond': 0.25   # Article 325w(3)(c)
    # Add other LGDs if needed
}

# Risk Weights (Article 325y, Table 2)
rw_map = {
    'AAA': 0.005,
    'AA': 0.02,
    'A': 0.03,
    'BBB': 0.06,
    'BB': 0.15,
    'B': 0.30,
    'CCC': 0.50, # Includes CCC+ and below
    'Unrated': 0.15,
    'Defaulted': 1.00,
    # Add mapping for specific agency ratings if needed
    # For simplicity, we use the letter grade directly
}

print("--- Setup Complete --- ")
print("\nEquity Portfolio:")
display(equity_df)
print("\nDebt Portfolio (Senior):")
display(debt_senior_df)
print("\nDebt Portfolio (Non-Senior):")
display(debt_non_senior_df)

--- Setup Complete --- 

Equity Portfolio:


,Bucket,Issuer,Maturity,Asset_Class,Seniority,Rating,Exposure_VA
0,Corp,Issuer A,0,Equity,NaN,Unrated,697
1,Corp,Issuer A,0,Equity,NaN,Unrated,-1240
2,Corp,Issuer B,0,Equity,NaN,BBB,9843
3,Corp,Issuer B,0,Equity,NaN,BBB,-2659



Debt Portfolio (Senior):


,Bucket,Issuer,Scaled_Maturity_Factor,Asset_Class,Rating,Exposure_VA,Seniority
0,Sovereign,Issuer A,0,Debt,CCC,-93197,Senior
1,Sovereign,Issuer A,0,Debt,CCC,186394,Senior
2,Sovereign,Issuer B,1,Debt,AA,-1060061,Senior
3,Sovereign,Issuer B,1,Debt,AA,913941,Senior
4,Sovereign,Issuer C,1,Debt,AA,499947,Senior
5,Sovereign,Issuer C,1,Debt,AA,-270395,Senior



Debt Portfolio (Non-Senior):


,Bucket,Issuer,Scaled_Maturity_Factor,Asset_Class,Rating,Exposure_VA,Seniority
0,Sovereign,Issuer A,0,Debt,CCC,-93197,Non-Senior
1,Sovereign,Issuer A,0,Debt,CCC,186394,Non-Senior
2,Sovereign,Issuer B,1,Debt,AA,-1060061,Non-Senior
3,Sovereign,Issuer B,1,Debt,AA,913941,Non-Senior
4,Sovereign,Issuer C,1,Debt,AA,499947,Non-Senior
5,Sovereign,Issuer C,1,Debt,AA,-270395,Non-Senior


## Step 1: Identify Portfolio Positions

This step involves listing the positions and their relevant attributes (Bucket, Issuer, Maturity/Scaling Factor, Asset Class, Seniority, Rating, Exposure). The DataFrames created in Cell 1 represent this step.

In [9]:
# Cell 2: Step 2 - Calculate Gross JTD
# This step calculates the potential loss for each individual position upon issuer default, before any netting or scaling.
# Article 325w: Gross JTD = Exposure * LGD.
# LGD is 100% for Equity (Art 325w(4)), 75% for Senior Debt (Art 325w(3)(b)), 100% for Non-Senior Debt (Art 325w(3)(a)).
print("--- Step 2: Calculate Gross JTD ---")

def calculate_gross_jtd(df, lgd_map):
    """Calculates Gross JTD based on Article 325w."""
    df_calc = df.copy()

    # Determine LGD based on Asset Class and Seniority
    def get_lgd(row):
        if row['Asset_Class'] == 'Equity':
            return lgd_map['Equity']
        elif pd.notna(row['Seniority']) and row['Seniority'] in lgd_map:
            return lgd_map[row['Seniority']]
        else:
            # Fallback for unexpected cases
            print(f"Warning: Could not determine LGD for row: {row.name}")
            return np.nan

    df_calc['LGD'] = df_calc.apply(get_lgd, axis=1)

    # Calculate Gross JTD (Exposure_VA * LGD)
    # The JTD is the loss amount, so it preserves the sign of the exposure implicitly handling max/min from Art 325w(1,2)
    df_calc['Gross_JTD'] = (df_calc['Exposure_VA'] * df_calc['LGD']).round()

    # We don't strictly need VD for the calculation if JTD = VA * LGD, but can show it for clarity
    # df_calc['VD'] = df_calc['Exposure_VA'] * (1 - df_calc['LGD'])

    return df_calc

# Calculate Gross JTD for each portfolio
equity_step2 = calculate_gross_jtd(equity_df, lgd_map)
debt_senior_step2 = calculate_gross_jtd(debt_senior_df, lgd_map)
debt_non_senior_step2 = calculate_gross_jtd(debt_non_senior_df, lgd_map)

print("\nEquity Portfolio - Gross JTD Calculation:")
# Displaying relevant columns for verification
display(equity_step2[['Issuer', 'Exposure_VA', 'LGD', 'Gross_JTD']])

print("\nDebt Portfolio (Senior) - Gross JTD Calculation:")
display(debt_senior_step2[['Issuer', 'Exposure_VA', 'LGD', 'Gross_JTD']])

print("\nDebt Portfolio (Non-Senior) - Gross JTD Calculation:")
display(debt_non_senior_step2[['Issuer', 'Exposure_VA', 'LGD', 'Gross_JTD']])

--- Step 2: Calculate Gross JTD ---

Equity Portfolio - Gross JTD Calculation:


,Issuer,Exposure_VA,LGD,Gross_JTD
0,Issuer A,697,1,697
1,Issuer A,-1240,1,"-1,240"
2,Issuer B,9843,1,"9,843"
3,Issuer B,-2659,1,"-2,659"



Debt Portfolio (Senior) - Gross JTD Calculation:


,Issuer,Exposure_VA,LGD,Gross_JTD
0,Issuer A,-93197,1,"-69,898"
1,Issuer A,186394,1,"139,796"
2,Issuer B,-1060061,1,"-795,046"
3,Issuer B,913941,1,"685,456"
4,Issuer C,499947,1,"374,960"
5,Issuer C,-270395,1,"-202,796"



Debt Portfolio (Non-Senior) - Gross JTD Calculation:


,Issuer,Exposure_VA,LGD,Gross_JTD
0,Issuer A,-93197,1,"-93,197"
1,Issuer A,186394,1,"186,394"
2,Issuer B,-1060061,1,"-1,060,061"
3,Issuer B,913941,1,"913,941"
4,Issuer C,499947,1,"499,947"
5,Issuer C,-270395,1,"-270,395"


## Step 3: Calculate Net JTD (Offsetting and Scaling)

This step adjusts Gross JTD for maturity and nets positions for the same issuer (Article 325x).
1. **Maturity Scaling:** Apply the 'Scaling_Factor' (derived from 'Maturity' for Equity or using 'Scaled_Maturity_Factor' for Debt) to the Gross JTD. The factor is `max(Maturity, 0.25)` if Maturity < 1, else 1.0. (Art 325x(2)(b), 325x(3)).
2. **Netting:** Group by issuer and sum the 'Scaled_JTD' values. Seniority conditions (Art 325x(1)) are met as equity is uniform and debt examples use consistent seniority.

In [10]:
# Cell 3: Step 3 - Calculate Net JTD
print("--- Step 3: Calculate Net JTD ---")

def calculate_net_jtd(df):
    """Applies scaling and nets JTD by issuer according to Article 325x."""
    df_calc = df.copy()

    # Determine and Apply Scaling Factor
    if 'Maturity' in df_calc.columns: # For Equity portfolio
        # Factor is max(Maturity, 0.25) if Maturity < 1 year, else 1.0
        df_calc['Scaling_Factor'] = np.where(df_calc['Maturity'] < 1, np.maximum(df_calc['Maturity'], 0.25), 1.0)
    elif 'Scaled_Maturity_Factor' in df_calc.columns: # For Debt portfolio with pre-calculated factor
        df_calc['Scaling_Factor'] = df_calc['Scaled_Maturity_Factor']
    else:
        print("Warning: No Maturity or Scaled_Maturity_Factor column found. Assuming scaling factor = 1.0")
        df_calc['Scaling_Factor'] = 1.0

    # Calculate Scaled JTD by applying the factor to Gross JTD
    df_calc['Scaled_JTD'] = (df_calc['Gross_JTD'] * df_calc['Scaling_Factor']).round()

    # Netting by Issuer: Group by 'Issuer' and sum the 'Scaled_JTD'
    # We create a new DataFrame holding the results per issuer
    df_net_grouped = df_calc.groupby('Issuer').agg(
        # Aggregate necessary info for later steps
        Bucket=('Bucket', 'first'),
        Rating=('Rating', 'first'),
        Net_JTD=('Scaled_JTD', 'sum')
    ).reset_index().set_index('Issuer') # Use Issuer as index for clarity

    # Display intermediate Scaled JTDs before netting for verification
    print("\nIntermediate Scaled JTDs (before netting):")
    display(df_calc[['Issuer', 'Gross_JTD', 'Scaling_Factor', 'Scaled_JTD']])

    # Return the DataFrame with one row per issuer, containing the Net_JTD
    return df_net_grouped


# --- Calculate Net JTD for each portfolio ---
print("\n--- Equity Net JTD Calculation ---")
equity_step3 = calculate_net_jtd(equity_step2)
print("\nEquity Portfolio - Final Net JTD per Issuer:")
display(equity_step3)

print("\n--- Debt (Senior) Net JTD Calculation ---")
debt_senior_step3 = calculate_net_jtd(debt_senior_step2)
print("\nDebt Portfolio (Senior) - Final Net JTD per Issuer:")
display(debt_senior_step3)

print("\n--- Debt (Non-Senior) Net JTD Calculation ---")
debt_non_senior_step3 = calculate_net_jtd(debt_non_senior_step2)
print("\nDebt Portfolio (Non-Senior) - Final Net JTD per Issuer:")
display(debt_non_senior_step3)

--- Step 3: Calculate Net JTD ---

--- Equity Net JTD Calculation ---

Intermediate Scaled JTDs (before netting):


,Issuer,Gross_JTD,Scaling_Factor,Scaled_JTD
0,Issuer A,697,0,174
1,Issuer A,"-1,240",0,-310
2,Issuer B,"9,843",0,"2,461"
3,Issuer B,"-2,659",0,-665



Equity Portfolio - Final Net JTD per Issuer:


,Bucket,Rating,Net_JTD
Issuer,,,
Issuer A,Corp,Unrated,-136
Issuer B,Corp,BBB,"1,796"



--- Debt (Senior) Net JTD Calculation ---

Intermediate Scaled JTDs (before netting):


...

,Bucket,Rating,Net_JTD
Issuer,,,
Issuer A,Sovereign,CCC,"17,475"
Issuer B,Sovereign,AA,"73,271"
Issuer C,Sovereign,AA,"172,164"



--- Debt (Non-Senior) Net JTD Calculation ---

Intermediate Scaled JTDs (before netting):


,Issuer,Gross_JTD,Scaling_Factor,Scaled_JTD
0,Issuer A,"-93,197",0,"-23,299"
1,Issuer A,"186,394",0,"46,598"
2,Issuer B,"-1,060,061",1,"-816,247"
3,Issuer B,"913,941",1,"913,941"
4,Issuer C,"499,947",1,"499,947"
5,Issuer C,"-270,395",1,"-270,395"



Debt Portfolio (Non-Senior) - Final Net JTD per Issuer:


,Bucket,Rating,Net_JTD
Issuer,,,
Issuer A,Sovereign,CCC,"23,299"
Issuer B,Sovereign,AA,"97,694"
Issuer C,Sovereign,AA,"229,552"


## Step 4: Apply Default Risk Weights

Multiply the Net JTD for each issuer by the regulatory risk weight corresponding to its rating (Article 325y, Table 2). This converts the net exposure into a risk-weighted capital amount.

In [11]:
# Cell 4: Step 4 - Apply Risk Weights
print("--- Step 4: Apply Default Risk Weights ---")

def apply_risk_weights(df_net_jtd, rw_map):
    """Applies risk weights based on rating to the Net JTD DataFrame."""
    df_calc = df_net_jtd.copy()
    # Map rating to risk weight using the rw_map dictionary
    df_calc['Risk_Weight'] = df_calc['Rating'].map(rw_map)
    # Handle potential missing ratings (e.g., if a rating isn't in rw_map)
    # We assign the 'Unrated' risk weight as a fallback
    df_calc['Risk_Weight'].fillna(rw_map.get('Unrated', 0.15), inplace=True)

    # Calculate Weighted Net JTD = Net JTD * Risk Weight
    df_calc['Weighted_Net_JTD'] = (df_calc['Net_JTD'] * df_calc['Risk_Weight']).round()
    return df_calc

# Calculate Weighted Net JTD for each portfolio
equity_step4 = apply_risk_weights(equity_step3, rw_map)
debt_senior_step4 = apply_risk_weights(debt_senior_step3, rw_map)
debt_non_senior_step4 = apply_risk_weights(debt_non_senior_step3, rw_map)

print("\nEquity Portfolio - Weighted Net JTD per Issuer:")
# Display relevant columns for verification
display(equity_step4[['Bucket', 'Rating', 'Net_JTD', 'Risk_Weight', 'Weighted_Net_JTD']])

print("\nDebt Portfolio (Senior) - Weighted Net JTD per Issuer:")
display(debt_senior_step4[['Bucket', 'Rating', 'Net_JTD', 'Risk_Weight', 'Weighted_Net_JTD']])

print("\nDebt Portfolio (Non-Senior) - Weighted Net JTD per Issuer:")
display(debt_non_senior_step4[['Bucket', 'Rating', 'Net_JTD', 'Risk_Weight', 'Weighted_Net_JTD']])

--- Step 4: Apply Default Risk Weights ---

Equity Portfolio - Weighted Net JTD per Issuer:


/tmp/ipython-input-3988459990.py:11: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_calc['Risk_Weight'].fillna(rw_map.get('Unrated', 0.15), inplace=True)
/tmp/ipython-input-3988459990.py:11: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(val

,Bucket,Rating,Net_JTD,Risk_Weight,Weighted_Net_JTD
Issuer,,,,,
Issuer A,Corp,Unrated,-136,0,-20
Issuer B,Corp,BBB,"1,796",0,108



Debt Portfolio (Senior) - Weighted Net JTD per Issuer:


,Bucket,Rating,Net_JTD,Risk_Weight,Weighted_Net_JTD
Issuer,,,,,
Issuer A,Sovereign,CCC,"17,475",0,"8,738"
Issuer B,Sovereign,AA,"73,271",0,"1,465"
Issuer C,Sovereign,AA,"172,164",0,"3,443"



Debt Portfolio (Non-Senior) - Weighted Net JTD per Issuer:


,Bucket,Rating,Net_JTD,Risk_Weight,Weighted_Net_JTD
Issuer,,,,,
Issuer A,Sovereign,CCC,"23,299",0,"11,650"
Issuer B,Sovereign,AA,"97,694",0,"1,954"
Issuer C,Sovereign,AA,"229,552",0,"4,591"


## Step 5: Calculate Hedge Benefit Ratio (WtS)

Calculate the WtS ratio for each bucket using the *unweighted* Net JTD values from Step 3 (Article 325y(4)). This ratio determines how much hedging benefit is recognized in the final aggregation.

`WtS = Sum(Net JTD Long) / ( Sum(Net JTD Long) + Sum(|Net JTD Short|) )`

In [12]:
# Cell 5: Step 5 - Calculate WtS per Bucket
print("--- Step 5: Calculate Hedge Benefit Ratio (WtS) ---")

# Input DataFrame for this function is the result of Step 4 (df_step4)
# It contains 'Bucket' and 'Net_JTD' (unweighted) needed for WtS calculation
def calculate_wts(df_step4):
    """Calculates WtS per bucket using Net JTD (unweighted)."""
    # Group by bucket to calculate WtS for each bucket separately
    bucket_groups = df_step4.groupby('Bucket')
    wts_results = {}

    print("Calculating WtS Ratios per Bucket:")
    # Iterate through each bucket group
    for bucket_name, group_df in bucket_groups:
        # Sum of positive Net JTDs (long positions) in the bucket
        net_jtd_long_sum = group_df[group_df['Net_JTD'] > 0]['Net_JTD'].sum()
        # Sum of the absolute values of negative Net JTDs (short positions) in the bucket
        net_jtd_short_abs_sum = group_df[group_df['Net_JTD'] < 0]['Net_JTD'].abs().sum()

        # Calculate the denominator for the WtS formula
        denominator = net_jtd_long_sum + net_jtd_short_abs_sum

        # Calculate WtS, handle division by zero if bucket has zero net exposure
        if denominator == 0:
            wts = 1.0 # Default WtS is 1 if bucket is flat or empty
        else:
            wts = net_jtd_long_sum / denominator

        # Store the WtS result for the bucket
        wts_results[bucket_name] = wts
        # Print the results for verification
        print(f"  Bucket '{bucket_name}': Sum Net JTD Long={net_jtd_long_sum:,.0f}, Sum |Net JTD Short|={net_jtd_short_abs_sum:,.0f}, WtS={wts:.4f}")

    return wts_results # Return dictionary {bucket_name: wts_value}

# --- Calculate WtS for each portfolio ---
print("\n--- Equity WtS ---")
equity_wts = calculate_wts(equity_step4) # Pass result of Step 4

print("\n--- Debt (Senior) WtS ---")
debt_senior_wts = calculate_wts(debt_senior_step4) # Pass result of Step 4

print("\n--- Debt (Non-Senior) WtS ---")
debt_non_senior_wts = calculate_wts(debt_non_senior_step4) # Pass result of Step 4

--- Step 5: Calculate Hedge Benefit Ratio (WtS) ---

--- Equity WtS ---
Calculating WtS Ratios per Bucket:
  Bucket 'Corp': Sum Net JTD Long=1,796, Sum |Net JTD Short|=136, WtS=0.9296

--- Debt (Senior) WtS ---
Calculating WtS Ratios per Bucket:
  Bucket 'Sovereign': Sum Net JTD Long=262,910, Sum |Net JTD Short|=0, WtS=1.0000

--- Debt (Non-Senior) WtS ---
Calculating WtS Ratios per Bucket:
  Bucket 'Sovereign': Sum Net JTD Long=350,545, Sum |Net JTD Short|=0, WtS=1.0000


## Step 6: Aggregate Within the Bucket

Apply the final DRC aggregation formula using the Risk-Weighted JTDs (from Step 4) and the WtS (from Step 5) for each bucket (Article 325y(4)).

`DRC_Bucket = max( Sum(Weighted Net JTD Long) - WtS * Sum(|Weighted Net JTD Short|) , 0 )`

The total DRC is the sum of the charges across all buckets (Article 325y(5)).

In [13]:
# Cell 6: Step 6 - Aggregate Bucket DRC
print("--- Step 6: Aggregate Within Bucket and Calculate Total DRC --- ")

# Input DataFrame for this function is the result of Step 4 (df_step4)
# Input wts_map is the result of Step 5
def calculate_bucket_drc(df_step4, wts_map):
    """Calculates the final DRC per bucket and the total DRC."""
    # Group by bucket to calculate DRC for each bucket separately
    bucket_groups = df_step4.groupby('Bucket')
    drc_results_per_bucket = {}

    print("Calculating Bucket DRCs:")
    # Iterate through each bucket group
    for bucket_name, group_df in bucket_groups:
        # Retrieve the WtS for the current bucket
        wts = wts_map.get(bucket_name, 1.0) # Default to 1.0 if bucket somehow not in wts_map

        # Sum of positive Weighted Net JTDs (long risk-weighted exposure) in the bucket
        weighted_jtd_long_sum = group_df[group_df['Weighted_Net_JTD'] > 0]['Weighted_Net_JTD'].sum()
        # Sum of the absolute values of negative Weighted Net JTDs (short risk-weighted exposure) in the bucket
        weighted_jtd_short_abs_sum = group_df[group_df['Weighted_Net_JTD'] < 0]['Weighted_Net_JTD'].abs().sum()

        # Apply the DRC formula from Article 325y(4)
        # DRC_Bucket = max( Sum(Weighted Longs) - WtS * Sum(|Weighted Shorts|) , 0 )
        drc_bucket = max(weighted_jtd_long_sum - wts * weighted_jtd_short_abs_sum, 0)

        # Store the DRC result for the bucket
        drc_results_per_bucket[bucket_name] = drc_bucket
        # Print the calculation details for verification
        print(f"  Bucket '{bucket_name}': Sum Weighted Longs={weighted_jtd_long_sum:,.0f}, Sum |Weighted Shorts|={weighted_jtd_short_abs_sum:,.0f}, WtS={wts:.4f} => DRC={drc_bucket:,.0f}")

    # Calculate the Total Non-Securitisation DRC by summing the DRC of all buckets (Article 325y(5))
    total_drc = sum(drc_results_per_bucket.values())
    print(f"\n---> Total Non-Securitisation DRC = {total_drc:,.0f}")

    # Return dictionary of bucket DRCs and the total DRC
    return drc_results_per_bucket, total_drc

# --- Calculate Final DRC for each portfolio ---
print("\n------ Equity Final DRC ------")
equity_drc_buckets, equity_total_drc = calculate_bucket_drc(equity_step4, equity_wts)

print("\n------ Debt (Senior) Final DRC ------")
debt_senior_drc_buckets, debt_senior_total_drc = calculate_bucket_drc(debt_senior_step4, debt_senior_wts)

print("\n------ Debt (Non-Senior) Final DRC ------")
debt_non_senior_drc_buckets, debt_non_senior_total_drc = calculate_bucket_drc(debt_non_senior_step4, debt_non_senior_wts)

--- Step 6: Aggregate Within Bucket and Calculate Total DRC --- 

------ Equity Final DRC ------
Calculating Bucket DRCs:
  Bucket 'Corp': Sum Weighted Longs=108, Sum |Weighted Shorts|=20, WtS=0.9296 => DRC=89

---> Total Non-Securitisation DRC = 89

------ Debt (Senior) Final DRC ------
Calculating Bucket DRCs:
  Bucket 'Sovereign': Sum Weighted Longs=13,646, Sum |Weighted Shorts|=0, WtS=1.0000 => DRC=13,646

---> Total Non-Securitisation DRC = 13,646

------ Debt (Non-Senior) Final DRC ------
Calculating Bucket DRCs:
  Bucket 'Sovereign': Sum Weighted Longs=18,195, Sum |Weighted Shorts|=0, WtS=1.0000 => DRC=18,195

---> Total Non-Securitisation DRC = 18,195
